In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd

In [ ]:
def find_drawdowns_and_recoveries(ticker="VTI", start="2000-01-01", end=None, min_dd_pct=5.0, min_recovery_pct=0.0):
    """
    Find all major drawdowns and recoveries for a given ticker.

    Args:
        ticker: Stock/ETF ticker (default: VTI)
        start: Start date (default: 2000-01-01)
        end: End date (default: today)
        min_dd_pct: Minimum drawdown threshold in % (default: 5%)
        min_recovery_pct: Percent above the prior peak required to count as recovered (default: 0%)

    Returns:
        Dictionary with drawdown events and recovery points
    """

    # Robust handling of 'today' or None for end date
    import datetime
    if end is None or (isinstance(end, str) and end.lower() == 'today'):
        end_dt = datetime.date.today().strftime('%Y-%m-%d')
    else:
        end_dt = end

    # Fetch price data
    print(f"Fetching {ticker} data from {start} to {end_dt}...")
    df = yf.download(ticker, start=start, end=end_dt, progress=False)
    prices = df['Close'].copy()

    if prices.empty:
        print(f"No data found for {ticker}")
        return None

    # Convert to numpy arrays for easier iteration
    dates = prices.index.to_numpy()
    values = prices.values

    # Calculate running maximum
    running_max = np.maximum.accumulate(values)

    # Calculate drawdown percentages
    drawdown_pct = (values / running_max - 1.0) * 100

    results = {
        'ticker': ticker,
        'fetch_start': start,
        'fetch_end': end_dt,
        'data_period': f"{prices.index[0].date()} to {prices.index[-1].date()}",
        'drawdown_events': []
    }

    # Find peaks and troughs
    drawdown_events = []
    in_drawdown = False
    peak_idx = 0
    last_peak_search_start = 0  # ensures next peak is after prior recovery threshold

    for i in range(1, len(drawdown_pct)):
        dd_now = drawdown_pct[i]

        # Entering a new drawdown
        if not in_drawdown and dd_now < -min_dd_pct:
            in_drawdown = True
            # Find the peak since the last recovery threshold, not the all-time peak
            peak_idx = last_peak_search_start + np.argmax(values[last_peak_search_start:i+1])

        # Find trough while in drawdown
        if in_drawdown:
            # Check if we've recovered back to the required threshold
            recovery_threshold = values[peak_idx] * (1 + min_recovery_pct / 100)
            if values[i] >= recovery_threshold:
                in_drawdown = False

                # Record the event
                peak_date = dates[peak_idx]
                peak_price = values[peak_idx]

                # Find minimum after peak up to current point
                min_idx = peak_idx + np.argmin(values[peak_idx:i+1])
                trough_date = dates[min_idx]
                trough_price = values[min_idx]

                dd_magnitude = (trough_price / peak_price - 1.0) * 100

                if dd_magnitude <= -min_dd_pct:
                    recovery_idx = i
                    recovery_date = dates[recovery_idx]
                    recovery_price = values[recovery_idx]

                    # Convert numpy datetime64 to Python datetime
                    peak_dt = pd.Timestamp(peak_date).to_pydatetime().date()
                    trough_dt = pd.Timestamp(trough_date).to_pydatetime().date()
                    recovery_dt = pd.Timestamp(recovery_date).to_pydatetime().date()

                    # Calculate days using Python datetime differences
                    duration_to_trough = (trough_dt - peak_dt).days if peak_dt and trough_dt else None
                    recovery_days = (recovery_dt - trough_dt).days if recovery_dt and trough_dt else None

                    drawdown_events.append({
                        'event_num': len(drawdown_events) + 1,
                        'peak_date': peak_dt,
                        'peak_price': round(float(peak_price), 2),
                        'trough_date': trough_dt,
                        'trough_price': round(float(trough_price), 2),
                        'drawdown_pct': round(float(dd_magnitude), 2),
                        'recovery_date': recovery_dt,
                        'recovery_price': round(float(recovery_price), 2),
                        'recovery_days': recovery_days,
                        'duration_to_trough_days': duration_to_trough,
                        'recovery_threshold_pct': 100 + min_recovery_pct,
                    })

                # Set the next peak search window start at the recovery point
                last_peak_search_start = i

    results['drawdown_events'] = drawdown_events
    return results

In [ ]:
from IPython.display import display, Markdown

In [ ]:
# --- Compounding/Underwater Stats Utilities ---
def compounding_underwater_stats(prices, drawdown_pct, running_max, dates, min_dd_pct=0.0, min_recovery_pct=0.0):
    """
    Compute stats for underwater (drawdown below min_dd_pct and recovery above min_recovery_pct) and above-water periods.
    If both min_dd_pct and min_recovery_pct are 0, uses simple below-peak/above-peak logic.
    Returns:
        dict with underwater and above-water stats.
    """
    import numpy as np
    import pandas as pd

    if min_dd_pct == 0.0 and min_recovery_pct == 0.0:
        # Simple below-peak/above-peak logic
        underwater = prices < running_max
        above_water = ~underwater
        # Find contiguous underwater regions
        uw_periods = []
        aw_periods = []
        start = None
        for i, uw in enumerate(underwater):
            if uw and start is None:
                start = i
            elif not uw and start is not None:
                uw_periods.append((start, i-1))
                start = None
        if start is not None:
            uw_periods.append((start, len(underwater)-1))
        # Find contiguous above-water regions
        start = None
        for i, aw in enumerate(above_water):
            if aw and start is None:
                start = i
            elif not aw and start is not None:
                aw_periods.append((start, i-1))
                start = None
        if start is not None:
            aw_periods.append((start, len(above_water)-1))
    else:
        # Use drawdown/recovery logic similar to main function
        uw_periods = []
        aw_periods = []
        in_drawdown = False
        peak_idx = 0
        last_peak_search_start = 0
        i = 1
        while i < len(prices):
            dd_now = (prices[i] / running_max[i] - 1.0) * 100
            # Entering a new drawdown
            if not in_drawdown and dd_now < -min_dd_pct:
                in_drawdown = True
                peak_idx = last_peak_search_start + np.argmax(prices[last_peak_search_start:i+1])
                uw_start = peak_idx
            # Find trough while in drawdown
            if in_drawdown:
                recovery_threshold = prices[peak_idx] * (1 + min_recovery_pct / 100)
                if prices[i] >= recovery_threshold:
                    in_drawdown = False
                    uw_end = i-1
                    uw_periods.append((uw_start, uw_end))
                    last_peak_search_start = i
            i += 1
        # If still in drawdown at end
        if in_drawdown:
            uw_periods.append((uw_start, len(prices)-1))
        # Above-water periods are gaps between underwater
        prev_end = 0
        for uw_start, uw_end in uw_periods:
            if prev_end < uw_start:
                aw_periods.append((prev_end, uw_start-1))
            prev_end = uw_end+1
        if prev_end < len(prices):
            aw_periods.append((prev_end, len(prices)-1))

    # Underwater stats
    uw_lengths = [j-i+1 for i,j in uw_periods]
    # For each underwater period, find the minimum drawdown (trough) in that period
    uw_depths = [np.min(drawdown_pct[i:j+1]) for i,j in uw_periods]
    uw_ttr = [j-i+1 for i,j in uw_periods]  # time to recovery

    # Above-water stats
    aw_lengths = [j-i+1 for i,j in aw_periods]
    aw_slopes = [(prices[j]-prices[i])/max(j-i,1) for i,j in aw_periods]

    total = len(prices)
    uw_total = sum(uw_lengths)
    aw_total = sum(aw_lengths)

    stats = {
        'underwater': {
            'num_periods': len(uw_periods),
            'avg_length': np.mean(uw_lengths) if uw_lengths else 0,
            'max_depth': np.min(uw_depths) if uw_depths else 0,  # most negative trough
            'avg_ttr': np.mean(uw_ttr) if uw_ttr else 0,
            'percent_time': 100*uw_total/total if total else 0,
        },
        'above_water': {
            'num_periods': len(aw_periods),
            'avg_length': np.mean(aw_lengths) if aw_lengths else 0,
            'avg_slope': np.mean(aw_slopes) if aw_slopes else 0,
            'percent_time': 100*aw_total/total if total else 0,
        }
    }
    return stats, uw_periods, aw_periods

def print_compounding_stats(prices, drawdown_pct, running_max, dates, min_dd_pct=0.0, min_recovery_pct=0.0):
    stats, uw_periods, aw_periods = compounding_underwater_stats(prices, drawdown_pct, running_max, dates, min_dd_pct, min_recovery_pct)
    import pandas as pd
    from IPython.display import display, Markdown

    display(Markdown('## Compounding/Underwater Statistics'))
    uw = stats['underwater']
    aw = stats['above_water']
    display(Markdown(f"**Underwater (Drawdown) Periods:**<br>"
                     f"- Number: {uw['num_periods']}<br>"
                     f"- Avg Length: {uw['avg_length']:.1f} days<br>"
                     f"- Max Depth (worst trough): {uw['max_depth']:.2f}%<br>"
                     f"- Avg TTR: {uw['avg_ttr']:.1f} days<br>"
                     f"- % Time: {uw['percent_time']:.1f}%"))
    display(Markdown(f"**Above Water (Recovery/New Highs) Periods:**<br>"
                     f"- Number: {aw['num_periods']}<br>"
                     f"- Avg Length: {aw['avg_length']:.1f} days<br>"
                     f"- Avg Slope: {aw['avg_slope']:.4f} per day<br>"
                     f"- % Time: {aw['percent_time']:.1f}%"))


In [ ]:
# --- Compounding/Underwater Stats Calculation and Display ---
# Run the analysis
results = find_drawdowns_and_recoveries(
    ticker="VOO",
    start="2020-01-01",
    end=None,  # Use today's date
    min_dd_pct=6,  # Find drawdowns of X% or more
    min_recovery_pct=3  # Require full X% recovery to prior peak
)

# Display results with nice formatting
print_drawdown_summary(results)

# --- Compounding/Underwater Stats ---
if results is not None:
    import yfinance as yf
    import numpy as np
    import pandas as pd
    df = yf.download(results['ticker'], start=results['fetch_start'], end=results['fetch_end'], progress=False)
    prices = df['Close'].copy()
    dates = prices.index.to_numpy()
    values = prices.values
    running_max = np.maximum.accumulate(values)
    # Use the same thresholds as the main analysis
    min_dd = results.get('min_dd_pct', 0) if 'min_dd_pct' in results else 6
    min_recovery = results.get('min_recovery_pct', 0) if 'min_recovery_pct' in results else 3
    drawdown_pct = (values / running_max - 1.0) * 100  # still needed for stats, but periods use thresholds
    print_compounding_stats(values, drawdown_pct, running_max, dates, min_dd_pct=min_dd, min_recovery_pct=min_recovery)


Fetching SPMO data from 2020-01-01 to 2026-01-30...


C:\Users\wongb\AppData\Local\Temp\ipykernel_26880\3269614571.py:100: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  'peak_price': round(float(peak_price), 2),
C:\Users\wongb\AppData\Local\Temp\ipykernel_26880\3269614571.py:102: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  'trough_price': round(float(trough_price), 2),
C:\Users\wongb\AppData\Local\Temp\ipykernel_26880\3269614571.py:103: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  'drawdown_pct': round(float(dd_mag

## Drawdown & Recovery Analysis: SPMO

**Period:** 2020-01-02 to 2026-01-29

,Event,Peak Date,Peak Price,Trough Date,Trough Price,DD %,Days to Trough,Recovery Date,Recovery Days
0,1,2020-02-19,$42.31,2020-03-23,$29.22,-30.95%,33,2020-07-22,121
1,2,2020-09-02,$49.39,2020-09-23,$44.83,-9.25%,21,2021-01-20,119
2,3,2021-02-12,$52.77,2021-03-08,$47.13,-10.67%,24,2021-06-22,106
3,4,2022-01-04,$62.36,2022-09-26,$48.18,-22.74%,265,2023-12-19,449
4,5,2024-03-22,$79.91,2024-04-19,$73.96,-7.45%,28,2024-06-05,47
5,6,2024-07-10,$90.67,2024-08-05,$78.74,-13.16%,26,2024-11-06,93
6,7,2025-02-13,$102.45,2025-04-04,$81.83,-20.13%,50,2025-06-02,59


### Summary Statistics

,Metric,Value
0,Total Major Drawdowns,7
1,Average Drawdown,-16.34%
2,Maximum Drawdown,-30.95%
3,Average Days to Trough,64 days
4,Average Recovery Days,142 days


## Compounding/Underwater Statistics

**Underwater (Drawdown) Periods:**<br>- Number: 8<br>- Avg Length: 131.9 days<br>- Max Depth (worst trough): -30.95%<br>- Avg TTR: 131.9 days<br>- % Time: 69.1%

**Above Water (Recovery/New Highs) Periods:**<br>- Number: 8<br>- Avg Length: 59.0 days<br>- Avg Slope: 0.1499 per day<br>- % Time: 30.9%